# 01 - Preprocessing and augmentation check

**What this notebook does**: verifies the input pipeline before any training happens -
resize, value range, label alignment, augmentation, and (again) that no file leaks between splits.

**What must already exist**: `outputs/splits/faithful_split.csv` and `outputs/splits/clean_split.csv`,
written by `00_dataset_audit.ipynb`. If they are missing, `load_split()` raises with instructions.

**Why this notebook matters**: every silent failure mode of the pipeline (wrong value range, labels
out of order, augmentation applied to the validation set) looks like a *model* problem later. Ten
minutes here saves a day of debugging a "bad model" that was actually a bad tensor.

**Key convention**: datasets yield **raw float32 pixels in [0, 255]**. Rescaling / `preprocess_input`
happens *inside* the model, so a model can never be paired with the wrong input scaling.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('img size :', IMG_SIZE, '| batch size:', BATCH_SIZE)
print('classes  :', CLASS_NAMES)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.train_utils import set_global_seeds, gpu_report

set_global_seeds(SEED)
print(gpu_report())

## 1. Load both split definitions

**Looks right**: `faithful` has ~1000 rows, `clean` has fewer (duplicates removed). Filepaths are
re-resolved against the current data root, so a split written locally still loads on Kaggle.

In [ ]:
from src.data_utils import load_split, split_counts

splits = {name: load_split(name) for name in SPLIT_VARIANTS}
for name, sdf in splits.items():
    print(f'--- {name} ({len(sdf)} images) ---')
    print(split_counts(sdf))
    print()

## 2. Re-verify no leakage into val/test

Runs the assertion again on the loaded CSVs (not just on the in-memory frames from notebook 00).

**Looks right**: `clean` PASSES, `faithful` reports leakage - that is expected and documented.

In [ ]:
from src.data_utils import assert_no_leakage

for name, sdf in splits.items():
    try:
        assert_no_leakage(sdf)
        print(f'{name:9s}: PASS - no hash spans two splits')
    except AssertionError as e:
        print(f'{name:9s}: leakage present -> {e}')

# Also confirm no identical *filepath* appears in two splits (a different bug class).
for name, sdf in splits.items():
    dupe_paths = sdf.groupby('filepath')['split'].nunique()
    print(f'{name:9s}: filepaths in >1 split:', int((dupe_paths > 1).sum()))

## 3. Build the tf.data pipelines (faithful split)

`make_split_datasets` shuffles + augments the training set only; val/test stay ordered so that
predictions line up with the dataframe's label column.

In [ ]:
from src.data_utils import make_split_datasets

train_ds, val_ds, test_ds, frames = make_split_datasets(splits['faithful'])
for name, ds in (('train', train_ds), ('val', val_ds), ('test', test_ds)):
    print(f'{name:5s}: {len(frames[name]):4d} images, {tf.data.experimental.cardinality(ds).numpy()} batches')

## 4. Inspect one batch

**Looks right**: shape `(32, 224, 224, 3)`, dtype `float32`, value range roughly `[0, 255]`
(NOT `[0, 1]` - the model rescales), labels are integers in `0..3`.

In [ ]:
images, labels = next(iter(train_ds))
print('image batch shape:', images.shape, '| dtype:', images.dtype)
print('pixel min/max/mean:', float(tf.reduce_min(images)), float(tf.reduce_max(images)),
      round(float(tf.reduce_mean(images)), 2))
print('label batch shape:', labels.shape, '| dtype:', labels.dtype)
print('labels in batch   :', sorted(set(labels.numpy().tolist())))
assert images.shape[1:] == (IMG_SIZE[0], IMG_SIZE[1], 3), 'unexpected image shape'
assert float(tf.reduce_max(images)) > 1.5, 'images look pre-scaled to [0,1] - the model rescales, so this is a bug'
print('\nOK - raw [0,255] float32 images of the expected shape.')

## 5. Confirm label order matches `CLASS_NAMES`

A silently permuted label mapping produces a plausible-looking but meaningless confusion matrix.

**Looks right**: for every row, the class name parsed from the file path equals
`CLASS_NAMES[label]`.

In [ ]:
from src.data_utils import normalize_class_name

check = splits['faithful'].copy()
check['from_path'] = [normalize_class_name(os.path.basename(os.path.dirname(p)))
                      for p in check['filepath']]
check['from_label'] = [CLASS_NAMES[i] for i in check['label']]
mismatch = check[check['from_path'] != check['from_label']]
print('rows checked   :', len(check))
print('label mismatches:', len(mismatch))
assert len(mismatch) == 0, 'label mapping is inconsistent with folder names'
print('OK - label indices agree with CLASS_NAMES order:', dict(enumerate(CLASS_NAMES)))

## 6. Save a grid of raw (unaugmented) samples

**Looks right**: recognisable chest CT slices, one row per class, all 224x224.

In [ ]:
from src.data_utils import make_dataset

sample_df = (splits['faithful'][splits['faithful']['split'] == 'train']
             .groupby('class', group_keys=False).head(4).reset_index(drop=True))
sample_ds = make_dataset(sample_df, batch_size=len(sample_df), shuffle=False, augment=False)
sample_images, sample_labels = next(iter(sample_ds))

fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(10, 10))
for ax, img, lab in zip(axes.ravel(), sample_images.numpy(), sample_labels.numpy()):
    ax.imshow(img.astype('uint8'))
    ax.set_title(CLASS_NAMES[int(lab)], fontsize=8)
    ax.axis('off')
fig.suptitle('Raw resized samples (224x224, no augmentation)')
fig.tight_layout()
out = FIGURES_DIR / 'samples_raw.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print('saved', out)

## 7. Save a grid of augmented samples

The same image passed through the augmenter several times.

**Looks right**: mild horizontal flips, small rotations/zooms/shifts. If the anatomy is unrecognisable
or images are heavily cropped, the augmentation is too aggressive for CT and should be toned down in
`data_utils.get_augmenter()`.

In [ ]:
from src.data_utils import get_augmenter

augmenter = get_augmenter(seed=SEED)
base = sample_images[:1]

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(base[0].numpy().astype('uint8'))
axes[0].set_title('original', fontsize=9)
axes[0].axis('off')
for ax in axes[1:]:
    aug = augmenter(base, training=True)[0].numpy()
    ax.imshow(np.clip(aug, 0, 255).astype('uint8'))
    ax.set_title('augmented', fontsize=9)
    ax.axis('off')
fig.tight_layout()
out = FIGURES_DIR / 'samples_augmented.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print('saved', out)

## 8. Confirm augmentation is NOT applied to val/test

Iterating the validation set twice must return byte-identical batches.

**Looks right**: `max abs difference: 0.0`. Anything else means augmentation or shuffling leaked into
evaluation, which would make every reported metric noisy and non-reproducible.

In [ ]:
first = next(iter(val_ds))[0].numpy()
second = next(iter(val_ds))[0].numpy()
diff = float(np.abs(first - second).max())
print('max abs difference between two passes over val_ds:', diff)
assert diff == 0.0, 'validation pipeline is not deterministic'

first_lab = next(iter(val_ds))[1].numpy()
expected = frames['val']['label'].values[:len(first_lab)]
print('first val labels match dataframe order:', bool((first_lab == expected).all()))

## 9. Repeat the pipeline checks on the `clean` split

**Looks right**: same shapes and value range; smaller counts; a visibly imbalanced train set
(`normal` smallest), which is why class weights are on by default here.

In [ ]:
from src.data_utils import imbalance_ratio
from src.train_utils import class_weights_for

c_train_ds, c_val_ds, c_test_ds, c_frames = make_split_datasets(splits['clean'])
c_images, c_labels = next(iter(c_train_ds))
print('clean train batch:', c_images.shape, c_images.dtype,
      '| pixel max:', float(tf.reduce_max(c_images)))
print('clean train class counts:')
print(c_frames['train']['class'].value_counts().reindex(CLASS_NAMES))
print('imbalance ratio:', round(imbalance_ratio(splits['clean'], 'train'), 2))
print('class_weight (clean) :', class_weights_for('clean', c_frames['train']['label'].values))
print('class_weight (faithful):', class_weights_for('faithful', frames['train']['label'].values))

## 10. Summary

If every assertion above passed, the input pipeline is sound and notebook 02 can train against it.

Checks performed:
1. both split definitions load and re-resolve their paths
2. `clean` is leakage-free; `faithful` leakage is present and documented
3. batches are `(B, 224, 224, 3)` float32 in `[0, 255]`
4. label indices agree with `CLASS_NAMES`
5. augmentation is applied to train only, and val/test iteration is deterministic

In [ ]:
print('preprocessing check complete.')
print('figures written to:', FIGURES_DIR)
print('next: 02_train_miniconvnet.ipynb')